# Task B — five-fold OOF evaluation of the current best recipe

This diagnostic notebook combines the original Task B training rows with the released labelled validation rows and evaluates the current best recipe: MuRIL TAPT, one reinitialized encoder layer, R-Drop 0.5, FGM, EMA, balanced class weights and six classifier epochs. It reports macro-F1, accuracy and a classification report from five held-out folds. It does not package a submission ZIP.

The default run uses one model seed (42) to keep the diagnostic practical. Set `SEEDS` to `[42, 43, 44, 45, 46]` for the full five-seed estimate used by the final submission recipe, at approximately five times the classifier cost. TAPT is run once on all combined comments, so this is a **transductive OOF** estimate: labels are held out from each fold, but the unlabelled wording is visible to the shared masked-language model.

In [ ]:
import os, pathlib, subprocess, sys, json
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", "origin", "task-b"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
raw_base = "https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/"
for rel in ["data/raw/hastika_multiclass_validation.csv", "data/raw/hastika_multiclass_test.csv"]:
    p = pathlib.Path(rel)
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        print("downloading missing released data:", rel)
        urlretrieve(raw_base + rel, p)
    assert p.exists() and p.stat().st_size > 0, f"missing required data: {rel}"
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score, classification_report
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Combine the labelled corpus

In [ ]:
from pathlib import Path
train_path = Path("data/raw/multiclass_train.csv")
val_path = Path("data/raw/hastika_multiclass_validation.csv")
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates("id", keep="first").reset_index(drop=True)
assert len(combined) == 3554 and combined["id"].is_unique
combined_path = Path("data/derived/task_b_oof_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
print("combined labelled rows:", len(combined))
LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
y = combined["Hate Category"].map(LABELS.index).to_numpy()


## 2. TAPT and held-out classifier fits

In [ ]:
SEEDS = [42]  # change to [42, 43, 44, 45, 46] for the full five-seed estimate
TAPT_OUT = "artifacts/runs/tapt-taskb-oof-trainval"
TAPT_LOG = "artifacts/logs/taskb_oof_trainval_tapt.log"
run([sys.executable, "-u", "-m", "hastika.task_b.tapt", "--model", "google/muril-base-cased", "--corpus", str(combined_path), "data/external/offenseval_kn.csv", "--val-frac", "0", "--min-words", "1", "--no-dedupe", "--epochs", "8", "--out", TAPT_OUT], log=TAPT_LOG)
assert (Path(TAPT_OUT) / "config.json").exists()
TAG = "b_oof_trainval_reinit1_rdrop"
run([sys.executable, "-u", "-m", "hastika.task_b.train", "--tag", TAG, "--model", TAPT_OUT, "--train-data", str(combined_path), "--folds", "5", "--no-dedupe", "--reinit-layers", "1", "--rdrop", "0.5", "--aux-weight", "0", "--seeds"] + [str(s) for s in SEEDS] + ["--epochs", "6"], log=f"artifacts/logs/{TAG}.log")
assert (Path("artifacts/runs") / TAG / "oof_probs.npy").exists()


## 3. Report the actual held-out performance

In [ ]:
probs = np.load(Path("artifacts/runs") / TAG / "oof_probs.npy")
assert probs.shape == (len(y), len(LABELS)), probs.shape
pred = probs.argmax(1)
macro = f1_score(y, pred, average="macro")
acc = accuracy_score(y, pred)
print(f"Task B transductive OOF macro-F1: {macro:.4f}")
print(f"Task B transductive OOF accuracy: {acc:.4f}")
print(classification_report(y, pred, target_names=LABELS, digits=4, zero_division=0))
OUT = Path("/kaggle/working/task_b_oof_outputs")
OUT.mkdir(parents=True, exist_ok=True)
np.save(OUT / "oof_probs.npy", probs)
pd.DataFrame({"metric": ["macro_f1", "accuracy"], "value": [macro, acc]}).to_csv(OUT / "summary.csv", index=False)
json.dump({"rows": len(y), "seeds": SEEDS, "transductive_tapt": True, "reinit_layers": 1, "rdrop": 0.5}, open(OUT / "config.json", "w"), indent=2)
print("DOWNLOAD RESULTS FROM:", OUT)
